# CPT_S 475 Semester Project
## Team: Project Hail Mary

**Team Members:**

*- Finley Blaylock*

*- Noah Manuel*

**Project Goal:**
Create three different machine-learning (ML) programs trained on the Kaggle [Kepler exoplanet dataset](https://www.kaggle.com/datasets/gauravkumar2525/kepler-exoplanet-dataset/data). 

Targeted ML programs:
- Decision Tree
- Random Forest Classifier
- Logistical Regression

## Part 0: Pre-requisites and Project Materials

In [ ]:
""" REQUIRED Packages"""
%pip install pandas numpy matplotlib seaborn scikit-learn

In [2]:
""" GLOBAL IMPORTS """

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb
import sklearn as skl

## Part 1: Data Cleansing

**Section Goal:** 
Take the dataset (located in `./data/exoplanets_data.csv`) and cleanse the data for any missing values and errors that can affect the models training and testing data. 

In [3]:
DATA_PATH = "./data/exoplanets_data.csv"
CLEANSED_DATA_PATH = "./data/cleansed_exoplanets_data.csv"


DROP_KOI_SCORE = True


rawData = pd.read_csv(DATA_PATH)
rawDataFrame = rawData.copy()

print("Raw shape: ", rawDataFrame.shape)
print("NaN cells:", int(rawDataFrame.isna().sum().sum()))
print("Duplicate Rows", rawDataFrame.duplicated().sum())
print("Target counts (0=False Positive, 1=Candidate, 2=Confirmed): ", rawDataFrame['koi_disposition'].value_counts().sort_index().to_string())

Raw shape:  (9564, 12)
NaN cells: 0
Duplicate Rows 0
Target counts (0=False Positive, 1=Candidate, 2=Confirmed):  koi_disposition
0    4839
1    1982
2    2743


In [4]:
# Restorinng the original data frame to the raw data frame
"""
There are no NaN values in the dataset. However, there is a chance that these missing values were filled with the mean
of the column as a placeholder. Therefore, we have to restore these placedholder values with NaN to ensure that the training data isn't misrepresented. The following code will restore the NaN values in the dataset to ensure that the  data is 
accurate and representative of the actual data.
"""

FEATURES = ["koi_score", "koi_prad", "koi_teq", "koi_insol", "koi_steff", "koi_srad", "koi_slogg"]
CORE = ["koi_prad", "koi_teq", "koi_steff", "koi_srad", "koi_slogg"]

for c in FEATURES:
    # 1. Calculate the mean of the column
    col_mean = rawDataFrame[c].mean()
    
    # 2. Find where values are close to the mean (returns a boolean NumPy array)
    is_placeholder = np.isclose(rawDataFrame[c], col_mean, rtol=0, atol=1e-9)
    
    # 3. Use .values to guarantee a clean array for .loc and avoid Series hashing issues
    rawDataFrame.loc[is_placeholder, c] = np.nan

print("Number of NaN cells after restoring placeholders: ", int(rawDataFrame.isna().sum().sum()))

print("Shape of the DataFrame after reintroducing NaNs: ", rawDataFrame.shape)

Number of NaN cells after restoring placeholders:  3646
Shape of the DataFrame after reintroducing NaNs:  (9564, 12)


In [5]:
# Dropping unusable rows 

# Rows where every core planet/star measurement was a placeholder carry no information.
# They are 99% False Positive/Candidate, so keeping them would teach the models that "average values" = "not a planet".
no_measurements = rawDataFrame[CORE].isna().all(axis=1)
print("Rows with no core measurements:", no_measurements.sum())
print(rawDataFrame.loc[no_measurements, "koi_disposition"].value_counts().sort_index().to_string())
rawDataFrame = rawDataFrame.loc[~no_measurements]

# Physically invalid rows: Kepler observed for ~4 years, so a ~130,000-day period with zero
# insolation is a catalog error, not a planet.
invalid = (rawDataFrame["koi_insol"] <= 0) | (rawDataFrame["koi_period"] > 10000)
print("\nPhysically invalid rows:", invalid.sum(), "->", rawDataFrame.loc[invalid, "kepoi_name"].tolist())
rawDataFrame = rawDataFrame.loc[~invalid]

# extreme radii (100s-1000s of Earth radii) are kept. They are real eclipsing binaries
# labeled False Positive and carry signal; handle their skew with scaling/log transforms later.

print("Shape of dataframe after dropping unusable rows: ")
print(rawDataFrame.shape)

Rows with no core measurements: 363
koi_disposition
0    257
1    104
2      2

Physically invalid rows: 1 -> ['K01174.01']
Shape of dataframe after dropping unusable rows: 
(9200, 12)


In [6]:
# Final Dataframe shaping

if DROP_KOI_SCORE:
    rawDataFrame = rawDataFrame.drop(columns="koi_score")

# Identifiers are not features, but kepid is kept so Part 2 can split by star (several KOIs share one star)
rawDataFrame = rawDataFrame.reset_index(drop=True)

# Sanity checks
assert rawDataFrame.isna().sum().sum() == 0 or not DROP_KOI_SCORE, "unexpected NaNs remain"
assert rawDataFrame["kepoi_name"].is_unique
assert rawDataFrame["koi_disposition"].isin([0, 1, 2]).all()
assert (rawDataFrame[["koi_period", "koi_prad", "koi_teq", "koi_insol", "koi_steff", "koi_srad"]] > 0).all().all()

rawDataFrame.to_csv(CLEANSED_DATA_PATH, index=False)

summary = pd.DataFrame({"raw": rawDataFrame["koi_disposition"].value_counts().sort_index(),
                        "clean": rawDataFrame["koi_disposition"].value_counts().sort_index()})
summary["clean_%"] = (summary["clean"] / summary["clean"].sum() * 100).round(1)
print(f"Rows: {len(rawDataFrame)} -> {len(rawDataFrame)} ({len(rawDataFrame) - len(rawDataFrame)} removed) | columns: {rawDataFrame.shape[1]}")
print(f"Remaining NaN cells: {int(rawDataFrame.isna().sum().sum())}")
print(f"Saved to {CLEANSED_DATA_PATH}\n")
print(summary.to_string())

Rows: 9200 -> 9200 (0 removed) | columns: 11
Remaining NaN cells: 0
Saved to ./data/cleansed_exoplanets_data.csv

                  raw  clean  clean_%
koi_disposition                      
0                4582   4582     49.8
1                1877   1877     20.4
2                2741   2741     29.8


## Part 2: Training/Evaluation Data Creation

**Section Goal:** Split the dataset with a 70% training and 30% evaluation configuration. Each model will be trained on the same train-test dataset.

**Classifier Goal:** Predict `koi_disposition` (0 = False Positive, 1 = Candidate, 2 = Confirmed) from cleaned planet and star features. All models will use the same cleansed 70/30 train-test dataset so results are directly comprable. 

In [11]:
"""Section Specific Imports"""

from sklearn.model_selection import train_test_split

cleansedDataFrame = pd.read_csv(CLEANSED_DATA_PATH)

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    cleansedDataFrame.drop(columns=["koi_disposition", "kepid", "kepoi_name"]),
    cleansedDataFrame["koi_disposition"], 
    test_size=0.3, 
    random_state=42,
    stratify=cleansedDataFrame["koi_disposition"]
)

## Part 3: Decision Tree Classifier

**Section Goal:** Train a Decision Tree Classifier on the train-test dataset to evaluate performance.

**Why use Decision Tree:**

## Part 4: Random Forest Classifier

**Section Goal:** Train a Random Forest Classifier on the train-test dataset to evaluate performance.

**Why use Random Forest:**

## Part 5: Logistic Regression

**Section Goal:** Train a Logisitc Regression Classifier on the train-test dataset to evaluate performance.

**Why use Logistic Regression:**

## Part 6: Model Performance Comparison

**Section Goals:** 
- Compare the accuracy, precision, and false-positive (FP), false-negative (FN), true-negative (TN), true-positive (TP) rates of each model.
- Compare the strengths and weakness of each model 